In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
 
months = ["JANUARY","FEBRUARY","MARCH","APRIL","MAY","JUNE","JULY",
          "AUGUST","SEPTEMBER","OCTOBER","NOVEMBER","DECEMBER"]

time_slots = ["0-3 hrs. (Night)","3-6 hrs. (Night)","6-9 hrs (Day)","9-12 hrs (Day)",
              "12-15 hrs (Day)","15-18 hrs (Day)","18-21 hrs (Night)","21-24 hrs (Night)"]

In [3]:
spark = SparkSession.builder.appName("RoadAccidents").getOrCreate()

acc = spark.read.csv("file:///D:/BDA 20/ABD-BDA-LabWork/datasets/0909 Lab/Road_accidents.csv", header=True, inferSchema=True)
acc_time = spark.read.csv("file:///D:/BDA 20/ABD-BDA-LabWork/datasets/0909 Lab/Road_accidents_time.csv", header=True, inferSchema=True)
 
acc.printSchema()
acc.show(5)

root
 |-- STATE/UT: string (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- JANUARY: integer (nullable = true)
 |-- FEBRUARY: integer (nullable = true)
 |-- MARCH: integer (nullable = true)
 |-- APRIL: integer (nullable = true)
 |-- MAY: integer (nullable = true)
 |-- JUNE: integer (nullable = true)
 |-- JULY: integer (nullable = true)
 |-- AUGUST: integer (nullable = true)
 |-- SEPTEMBER: integer (nullable = true)
 |-- OCTOBER: integer (nullable = true)
 |-- NOVEMBER: integer (nullable = true)
 |-- DECEMBER: integer (nullable = true)

+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+
|     STATE/UT|YEAR|JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER|
+-------------+----+-------+--------+-----+-----+---+----+----+------+---------+-------+--------+--------+
|A & N Islands|2001|      8|      23|   15|   15| 14|  19|  14|    19|        7|     12|      13|      22|
|A & N Islands|2002

In [4]:
# 1. Add column 'Total', giving sum of accidents in each month . 
total_expr = sum(F.col(m) for m in months)
acc = acc.withColumn("Total", total_expr)
acc.select("STATE/UT", "YEAR", "Total").show(5)

+-------------+----+-----+
|     STATE/UT|YEAR|Total|
+-------------+----+-----+
|A & N Islands|2001|  181|
|A & N Islands|2002|  168|
|A & N Islands|2003|  180|
|A & N Islands|2004|  215|
|A & N Islands|2005|  206|
+-------------+----+-----+
only showing top 5 rows



In [5]:
# 2. Which state has highest number of accidents in year 2013? 
top_2013 = acc.filter(F.col("YEAR") == 2013).orderBy(F.desc("Total")).first()
print(f"Highest accidents in 2013: {top_2013['STATE/UT']} ({top_2013['Total']})")

Highest accidents in 2013: Tamil Nadu (66238)


In [6]:
# 3. Find the average monthly accidents for each state. 
avg_monthly = acc.groupBy("STATE/UT").agg(
    F.round(F.avg(total_expr / len(months)), 2).alias("avg_monthly_accidents")
).orderBy(F.desc("avg_monthly_accidents"))
avg_monthly.show(50, truncate=False)

+-----------------+---------------------+
|STATE/UT         |avg_monthly_accidents|
+-----------------+---------------------+
|Tamil Nadu       |5071.86              |
|Maharashtra      |3819.13              |
|Karnataka        |3480.72              |
|Andhra Pradesh   |3254.89              |
|Kerala           |3084.29              |
|Madhya Pradesh   |2345.28              |
|Rajasthan        |1926.51              |
|Gujarat          |1787.65              |
|Uttar Pradesh    |1634.1               |
|West Bengal      |1127.23              |
|Haryana          |785.24               |
|Delhi (Ut)       |711.31               |
|Chhattisgarh     |701.29               |
|Delhi Ut         |692.5                |
|Odisha           |685.93               |
|Bihar            |551.48               |
|Jammu & Kashmir  |492.98               |
|Assam            |367.37               |
|Punjab           |278.6                |
|Jharkhand        |275.05               |
|Goa              |267.45         

In [7]:
# 4. Which month has highest accidents? 
month_totals = acc.agg(*[F.sum(m).alias(m) for m in months]).first()
month_totals_dict = month_totals.asDict()
top_month = max(month_totals_dict, key=month_totals_dict.get)
print(f"Month with highest accidents: {top_month} ({month_totals_dict[top_month]})")

Month with highest accidents: MAY (521563)


In [17]:
# 5. Which time slot (like 0-3, 3-6 etc) has more accidents? What about 'Karnataka' state? 
  
slot_totals = acc_time.agg(*[F.sum(s).alias(s) for s in time_slots]).first()
slot_totals_dict = slot_totals.asDict()
top_slot = max(slot_totals_dict, key=slot_totals_dict.get)
print(f"Time slot with highest accidents (overall): {top_slot} ({slot_totals_dict[top_slot]})")

karnataka_time = acc_time.filter(F.col("STATE/UT") == "Karnataka")
karnataka_slot_totals = karnataka_time.agg(*[F.sum(s).alias(s) for s in time_slots]).first()
karnataka_slot_dict = karnataka_slot_totals.asDict()
top_slot_karnataka = max(karnataka_slot_dict, key=karnataka_slot_dict.get)
print(f"Time slot with highest accidents (Karnataka): {top_slot_karnataka} ({karnataka_slot_dict[top_slot_karnataka]})")

Time slot with highest accidents (overall): 15-18 (906639)
Time slot with highest accidents (Karnataka): 15-18 (93126)


In [12]:
# 6. Which state has more accidents from year 2001 to 2014? 
range_2001_2014 = acc.filter((F.col("YEAR") >= 2001) & (F.col("YEAR") <= 2014))
state_totals = range_2001_2014.groupBy("STATE/UT").agg(
    F.sum("Total").alias("total_accidents")
).orderBy(F.desc("total_accidents"))
top_state = state_totals.first()
print(f"State with most accidents 2001-2014: {top_state['STATE/UT']} ({top_state['total_accidents']})")

State with most accidents 2001-2014: Tamil Nadu (852073)


In [13]:
# 7. List states whose accidents number in year 2014 is less than state average from 2001 to 2014. 
state_avg_2001_2014 = range_2001_2014.groupBy("STATE/UT").agg(
    F.round(F.avg("Total"), 2).alias("avg_2001_2014")
)
year_2014 = acc.filter(F.col("YEAR") == 2014).select("STATE/UT", F.col("Total").alias("total_2014"))
 
below_avg = year_2014.join(state_avg_2001_2014, on="STATE/UT") \
    .filter(F.col("total_2014") < F.col("avg_2001_2014"))
below_avg.select("STATE/UT", "total_2014", "avg_2001_2014").show(50, truncate=False)

+-----------------+----------+-------------+
|STATE/UT         |total_2014|avg_2001_2014|
+-----------------+----------+-------------+
|Nagaland         |28        |55.79        |
|Kerala           |35872     |37011.5      |
|Daman & Diu      |39        |45.93        |
|Jammu & Kashmir  |5778      |5915.71      |
|Puducherry       |671       |1513.5       |
|Arunachal Pradesh|185       |242.07       |
|Sikkim           |130       |189.0        |
|Chandigarh       |366       |457.93       |
|Maharashtra      |44382     |45829.57     |
|Tripura          |716       |752.43       |
|Uttarakhand      |801       |1299.43      |
+-----------------+----------+-------------+



In [11]:
# 8. Which month has more accidents over the year?
year_month_totals = acc.groupBy("YEAR").agg(
    *[F.sum(m).alias(m) for m in months]
)
 
def top_month_for_row(row):
    d = row.asDict()
    year = d.pop("YEAR")
    top = max(d, key=d.get)
    return (year, top, d[top])
 
top_month_per_year = year_month_totals.rdd.map(top_month_for_row) \
    .toDF(["YEAR", "TOP_MONTH", "ACCIDENTS"]) \
    .orderBy("YEAR")
top_month_per_year.show(30, truncate=False)

+----+---------+---------+
|YEAR|TOP_MONTH|ACCIDENTS|
+----+---------+---------+
|2001|MAY      |29978    |
|2002|MAY      |31726    |
|2003|MAY      |30335    |
|2004|MAY      |31752    |
|2005|MAY      |40126    |
|2006|MAY      |37019    |
|2007|MAY      |39036    |
|2008|APRIL    |38059    |
|2009|MAY      |38928    |
|2010|MAY      |39821    |
|2011|MAY      |41542    |
|2012|MAY      |38920    |
|2013|MAY      |43064    |
|2014|MAY      |41404    |
+----+---------+---------+

